In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

BASE_DIR = Path(".")
DATA_PATH = BASE_DIR / "data" / "spread" / "raw.csv"
EDA_DIR = BASE_DIR / "eda"
FIG_DIR = EDA_DIR / "figures"
TABLE_DIR = EDA_DIR / "tables"
PER_PAIR_DIR = EDA_DIR / "per_pair"

for d in [EDA_DIR, FIG_DIR, TABLE_DIR, PER_PAIR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Input: {DATA_PATH}")
print(f"Output dir: {EDA_DIR.resolve()}")
print(f"Seaborn available: {HAS_SEABORN}")
print(f"Statsmodels available: {HAS_STATSMODELS}")

Input: data\spread\raw.csv
Output dir: C:\Users\vaibh\OneDrive\Documents\Projects\RL Pairs Trading\eda
Seaborn available: True
Statsmodels available: True


In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])

# Handle exported index columns from CSV writes
for c in ["Unnamed: 0", ""]:
    if c in df.columns:
        df = df.drop(columns=[c])

required_cols = ["Date", "Pair", "spread"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["spread"] = pd.to_numeric(df["spread"], errors="coerce")
df = df.sort_values(["Pair", "Date"]).reset_index(drop=True)

summary = pd.DataFrame(
    {
        "metric": ["rows", "pair_count", "date_min", "date_max", "missing_spread"],
        "value": [
            len(df),
            df["Pair"].nunique(),
            df["Date"].min().date(),
            df["Date"].max().date(),
            int(df["spread"].isna().sum()),
        ],
    }
)
summary.to_csv(TABLE_DIR / "dataset_summary.csv", index=False)

# Avoid hard dependency on tabulate
try:
    summary.to_markdown(TABLE_DIR / "dataset_summary.md", index=False)
except Exception:
    (TABLE_DIR / "dataset_summary.md").write_text(summary.to_string(index=False), encoding="utf-8")

summary

,metric,value
0,rows,24516
1,pair_count,12
2,date_min,2018-01-01
3,date_max,2026-04-10
4,missing_spread,2868


In [3]:
spread_clean = df["spread"].dropna()

global_stats = spread_clean.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame(name="spread")
global_stats.loc["skew"] = spread_clean.skew()
global_stats.loc["kurtosis"] = spread_clean.kurtosis()
global_stats.to_csv(TABLE_DIR / "global_spread_stats.csv")

fig, ax = plt.subplots(figsize=(10, 5))
if HAS_SEABORN:
    sns.histplot(spread_clean, bins=80, kde=True, ax=ax)
else:
    ax.hist(spread_clean, bins=80)
ax.set_title("Global Spread Distribution (Histogram + KDE)")
ax.set_xlabel("spread")
fig.tight_layout()
fig.savefig(FIG_DIR / "global_hist_kde.png", dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if HAS_SEABORN:
    sns.boxplot(x=spread_clean, ax=axes[0])
    sns.violinplot(x=spread_clean, ax=axes[1])
else:
    axes[0].boxplot(spread_clean, vert=False)
    axes[1].hist(spread_clean, bins=80)
axes[0].set_title("Global Spread Boxplot")
axes[1].set_title("Global Spread Violin")
fig.tight_layout()
fig.savefig(FIG_DIR / "global_box_violin.png", dpi=150)
plt.close(fig)

if HAS_STATSMODELS:
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111)
    sm.qqplot(spread_clean, line="45", ax=ax)
    ax.set_title("QQ Plot - Global Spread")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "global_qqplot.png", dpi=150)
    plt.close(fig)

global_stats.head(12)

,spread
count,21648.000000
mean,0.755447
std,1.295743
min,-2.278395
1%,-2.121748
5%,-1.943225
25%,-0.215330
50%,0.932828
75%,1.727655
95%,2.554980


In [4]:
daily = (
    df.groupby("Date", as_index=False)["spread"]
    .agg(mean_spread="mean", median_spread="median", std_spread="std")
    .sort_values("Date")
)
daily["rolling_std_30d"] = daily["std_spread"].rolling(30, min_periods=5).mean()
daily.to_csv(TABLE_DIR / "daily_cross_sectional_spread_stats.csv", index=False)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(daily["Date"], daily["mean_spread"], label="Mean", linewidth=1.5)
axes[0].plot(daily["Date"], daily["median_spread"], label="Median", linewidth=1.2)
axes[0].set_title("Cross-Sectional Spread Trend Over Time")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(daily["Date"], daily["std_spread"], label="Daily Std", linewidth=1)
axes[1].plot(daily["Date"], daily["rolling_std_30d"], label="30D Rolling Std", linewidth=1.5)
axes[1].set_title("Cross-Sectional Spread Dispersion")
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(FIG_DIR / "global_time_series.png", dpi=150)
plt.close(fig)

daily.head()

,Date,mean_spread,median_spread,std_spread,rolling_std_30d
0,2018-01-01,0.902878,1.223995,1.408835,NaN
1,2018-01-02,0.899921,1.215363,1.404052,NaN
2,2018-01-03,0.901246,1.216477,1.408032,NaN
3,2018-01-04,0.915134,1.231685,1.415908,NaN
4,2018-01-05,0.919155,1.244050,1.408272,1.40902


In [5]:
pair_stats = (
    df.groupby("Pair")["spread"]
    .agg(
        count="count",
        mean="mean",
        std="std",
        min="min",
        q01=lambda x: x.quantile(0.01),
        q05=lambda x: x.quantile(0.05),
        q25=lambda x: x.quantile(0.25),
        median="median",
        q75=lambda x: x.quantile(0.75),
        q95=lambda x: x.quantile(0.95),
        q99=lambda x: x.quantile(0.99),
        max="max",
        skew="skew",
        kurtosis=pd.Series.kurt,
    )
    .sort_values("std", ascending=False)
)

pair_stats.to_csv(TABLE_DIR / "per_pair_spread_stats.csv")

ranking_mean_high = pair_stats.sort_values("mean", ascending=False).head(20)
ranking_mean_low = pair_stats.sort_values("mean", ascending=True).head(20)
ranking_std_high = pair_stats.sort_values("std", ascending=False).head(20)

ranking_mean_high.to_csv(TABLE_DIR / "top20_pairs_by_mean_spread.csv")
ranking_mean_low.to_csv(TABLE_DIR / "bottom20_pairs_by_mean_spread.csv")
ranking_std_high.to_csv(TABLE_DIR / "top20_pairs_by_spread_std.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ranking_mean_high["mean"].sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Top 20 Pairs by Mean Spread")
axes[0].set_xlabel("mean spread")
ranking_std_high["std"].sort_values().plot(kind="barh", ax=axes[1], color="tab:orange")
axes[1].set_title("Top 20 Pairs by Spread Volatility")
axes[1].set_xlabel("std spread")
fig.tight_layout()
fig.savefig(FIG_DIR / "cross_pair_rankings.png", dpi=150)
plt.close(fig)

pair_stats.head()

,count,mean,std,min,q01,q05,q25,median,q75,q95,q99,max,skew,kurtosis
Pair,,,,,,,,,,,,,,
PNB.NS|BANKBARODA.NS,2043,1.173813,0.288987,0.544244,0.575925,0.653990,0.970740,1.259129,1.376244,1.541516,2.027361,2.109153,-0.139719,-0.011493
PNB.NS|CANBK.NS,2043,1.275425,0.250803,0.708448,0.746532,0.810019,1.155565,1.296636,1.427974,1.634745,2.076928,2.157467,0.074317,0.709634
HINDPETRO.NS|IOC.NS,2043,2.586464,0.219758,2.046394,2.257430,2.321713,2.422901,2.518377,2.697351,3.011895,3.056647,3.103709,0.741637,-0.527342
UNIONBANK.NS|CANBK.NS,2043,0.181424,0.181245,-0.265191,-0.221331,-0.154301,0.077028,0.181387,0.293470,0.489065,0.610315,0.652213,-0.038278,-0.095400
VBL.NS|DOMS.NS,569,2.436413,0.170487,2.116322,2.151894,2.217907,2.282057,2.406580,2.606022,2.707231,2.751462,2.764038,0.283607,-1.282808


In [6]:
LOOKBACK = 60  # causal rolling window; matches `preprocessing.build_binary_reversion_labels` / spread z logic


def rolling_z_preproc(spread: np.ndarray, L: int) -> np.ndarray:
    """Causal rolling z-score: z[i] = (spread[i] - mu) / sig using spread[i-L:i] (exclusive of i)."""
    n = len(spread)
    z = np.full(n, np.nan, dtype=np.float64)
    if n <= L:
        return z
    s = np.asarray(spread, dtype=np.float64)
    c = np.concatenate([[0.0], np.cumsum(s)])
    c2 = np.concatenate([[0.0], np.cumsum(s * s)])
    idx = np.arange(L, n, dtype=np.int64)
    sum_x = c[idx] - c[idx - L]
    sum_xx = c2[idx] - c2[idx - L]
    mu = sum_x / L
    den = max(L - 1, 1)
    var = np.maximum((sum_xx - sum_x * sum_x / L) / den, 0.0)
    sig = np.sqrt(np.maximum(var, 1e-16))
    z[idx] = (s[idx] - mu) / sig
    return z


# Per-pair causal z on sorted dates (requires df sorted by Pair, Date)
_df = df.sort_values(["Pair", "Date"]).reset_index(drop=True)
_df["z_60"] = _df.groupby("Pair", group_keys=False)["spread"].transform(
    lambda s: rolling_z_preproc(s.to_numpy(dtype=float), LOOKBACK)
)
df = _df
del _df

zfinite = df["z_60"].replace([np.inf, -np.inf], np.nan).dropna()
z_pair_summary = (
    df.groupby("Pair")["z_60"]
    .agg(
        n_finite="count",
        mean="mean",
        std="std",
        frac_abs_gt_1=lambda s: (s.abs() > 1).mean(),
        frac_abs_gt_2=lambda s: (s.abs() > 2).mean(),
        min="min",
        max="max",
    )
    .sort_values("std", ascending=False)
)
z_pair_summary.to_csv(TABLE_DIR / "per_pair_z60_summary.csv")

# Grid: all pairs z-score vs time
pairs_sorted = sorted(df["Pair"].unique())
n_pairs = len(pairs_sorted)
ncols = 4
nrows = int(np.ceil(n_pairs / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 2.4 * nrows), sharex=False)
axes = np.atleast_1d(axes).ravel()
for i, pair in enumerate(pairs_sorted):
    ax = axes[i]
    sub = df[df["Pair"] == pair].sort_values("Date")
    ax.plot(sub["Date"], sub["z_60"], linewidth=0.9, color="tab:blue")
    ax.axhline(0.0, color="black", linewidth=0.6)
    for z0, sty in [(1.0, "--"), (2.0, ":")]:
        ax.axhline(z0, color="tab:orange", linestyle=sty, linewidth=0.8, alpha=0.65)
        ax.axhline(-z0, color="tab:orange", linestyle=sty, linewidth=0.8, alpha=0.65)
    ax.set_title(pair, fontsize=8)
    ax.grid(alpha=0.2)
for j in range(len(pairs_sorted), len(axes)):
    axes[j].set_visible(False)
fig.suptitle(f"Rolling z-score of spread (L={LOOKBACK}, causal) — all pairs", fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "all_pairs_z60_timeseries_grid.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# Global z distribution (finite values)
fig, ax = plt.subplots(figsize=(10, 4.5))
if HAS_SEABORN:
    sns.histplot(zfinite, bins=80, kde=True, ax=ax, stat="density")
else:
    ax.hist(zfinite, bins=80, density=True)
ax.set_title(f"Global distribution of z_{{60}} (finite values, n={len(zfinite):,})")
ax.set_xlabel("z_60")
ax.axvline(0.0, color="black", linewidth=0.8)
fig.tight_layout()
fig.savefig(FIG_DIR / "global_z60_hist_kde.png", dpi=150)
plt.close(fig)

print(f"Added column z_60 (L={LOOKBACK}). Finite z rows: {len(zfinite):,} / {len(df):,}")
z_pair_summary.head(12)

Added column z_60 (L=60). Finite z rows: 19,830 / 24,516


,n_finite,mean,std,frac_abs_gt_1,frac_abs_gt_2,min,max
Pair,,,,,,,
JSWSTEEL.NS|TATASTEEL.NS,1983,0.111006,1.435274,0.546745,0.151248,-4.223333,5.006437
PNB.NS|BANKBARODA.NS,1983,-0.151242,1.432852,0.510524,0.153696,-4.361025,4.375821
HINDPETRO.NS|IOC.NS,1983,0.076116,1.428070,0.554087,0.143417,-7.038878,5.227442
TATASTEEL.NS|HINDALCO.NS,1983,-0.153505,1.425714,0.546745,0.147822,-4.368165,5.239381
PNB.NS|CANBK.NS,1983,-0.273148,1.419206,0.517376,0.141459,-5.819044,7.015112
BANKINDIA.NS|UNIONBANK.NS,1983,-0.032326,1.414102,0.465981,0.141948,-5.867357,6.703209
PFC.NS|RECLTD.NS,1983,0.199679,1.412649,0.519824,0.149780,-3.682654,4.911594
CANBK.NS|BANKBARODA.NS,1983,0.013275,1.412325,0.513950,0.138522,-4.262138,5.151707
UNIONBANK.NS|CANBK.NS,1983,-0.245837,1.369591,0.507097,0.136564,-4.506062,4.811217


In [7]:
for pair, pair_df in df.groupby("Pair"):
    safe_pair = pair.replace("/", "_").replace("\\", "_").replace("|", "__")
    pair_dir = PER_PAIR_DIR / safe_pair
    pair_dir.mkdir(parents=True, exist_ok=True)

    pair_df = pair_df.sort_values("Date")
    pair_spread = pair_df["spread"].dropna()

    # Time-series plot
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(pair_df["Date"], pair_df["spread"], linewidth=1)
    ax.set_title(f"Spread Time Series: {pair}")
    ax.set_xlabel("Date")
    ax.set_ylabel("spread")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(pair_dir / "timeseries.png", dpi=140)
    plt.close(fig)

    # Rolling z-score of spread (L=60, causal) — same definition as preprocessing.build_binary_reversion_labels
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(pair_df["Date"], pair_df["z_60"], linewidth=1, color="tab:blue")
    ax.axhline(0.0, color="black", linewidth=0.6)
    for z0, sty in [(1.0, "--"), (2.0, ":")]:
        ax.axhline(z0, color="tab:orange", linestyle=sty, linewidth=0.9, alpha=0.7)
        ax.axhline(-z0, color="tab:orange", linestyle=sty, linewidth=0.9, alpha=0.7)
    ax.set_title(f"Rolling z-score of spread (L={LOOKBACK}): {pair}")
    ax.set_xlabel("Date")
    ax.set_ylabel("z")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(pair_dir / "zscore_timeseries.png", dpi=140)
    plt.close(fig)

    # Distribution panel: histogram + boxplot
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    if HAS_SEABORN:
        sns.histplot(pair_spread, bins=45, kde=True, ax=axes[0])
        sns.boxplot(x=pair_spread, ax=axes[1])
    else:
        axes[0].hist(pair_spread, bins=45)
        axes[1].boxplot(pair_spread, vert=False)
    axes[0].set_title(f"Distribution: {pair}")
    axes[1].set_title(f"Boxplot: {pair}")
    fig.tight_layout()
    fig.savefig(pair_dir / "distribution.png", dpi=140)
    plt.close(fig)

print(f"Generated per-pair plots under: {PER_PAIR_DIR}")

Generated per-pair plots under: eda\per_pair


In [8]:
table_files = sorted([p.relative_to(EDA_DIR).as_posix() for p in TABLE_DIR.rglob("*") if p.is_file()])
figure_files = sorted([p.relative_to(EDA_DIR).as_posix() for p in FIG_DIR.rglob("*") if p.is_file()])
per_pair_dirs = sorted([p.name for p in PER_PAIR_DIR.iterdir() if p.is_dir()])

print("EDA export summary")
print("=" * 60)
print(f"Rows analyzed        : {len(df):,}")
print(f"Pairs analyzed       : {df['Pair'].nunique():,}")
print(f"Tables generated     : {len(table_files)}")
print(f"Figures generated    : {len(figure_files)}")
print(f"Per-pair folders     : {len(per_pair_dirs)}")
print("(Each pair folder also includes zscore_timeseries.png when z_60 cell was run.)")
print("\nSample tables:")
for f in table_files[:10]:
    print(f" - {f}")
print("\nSample figures:")
for f in figure_files[:10]:
    print(f" - {f}")

EDA export summary
Rows analyzed        : 24,516
Pairs analyzed       : 12
Tables generated     : 9
Figures generated    : 7
Per-pair folders     : 12
(Each pair folder also includes zscore_timeseries.png when z_60 cell was run.)

Sample tables:
 - tables/bottom20_pairs_by_mean_spread.csv
 - tables/daily_cross_sectional_spread_stats.csv
 - tables/dataset_summary.csv
 - tables/dataset_summary.md
 - tables/global_spread_stats.csv
 - tables/per_pair_spread_stats.csv
 - tables/per_pair_z60_summary.csv
 - tables/top20_pairs_by_mean_spread.csv
 - tables/top20_pairs_by_spread_std.csv

Sample figures:
 - figures/all_pairs_z60_timeseries_grid.png
 - figures/cross_pair_rankings.png
 - figures/global_box_violin.png
 - figures/global_hist_kde.png
 - figures/global_qqplot.png
 - figures/global_time_series.png
 - figures/global_z60_hist_kde.png
